In [1]:
# --- Setup: add project root to sys.path so that src is discoverable ---
import sys
from pathlib import Path
import datetime
import os
project_root = Path.cwd().parent  # assuming current working directory is 'notebook'
sys.path.insert(0, str(project_root))

# --- Create a timestamped results folder ---
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
results_folder = project_root / "results" / f"run_{timestamp}"
results_folder.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved in: {results_folder}")

# --- Imports from our modules ---
from src.config.loaders import load_config, load_station_data
from src.data.engineering import engineer_data
from src.data.scaling import scale_df, inverse_scale
from src.data.windowing import data_to_X_y
from src.visualization.plotting import plot_loss, plot_predictions

from src.models.lstm import build_original_simple_lstm_model, build_optimized_lstm_model
from src.models.cnn import build_cnn_model

# --- Standard Setup ---
import matplotlib.pyplot as plt
import os
import tensorflow as tf
import numpy as np
import pandas as pd

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = False
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

Results will be saved in: c:\Users\chris\OneDrive\Documents\Academic\CS 370\tx-soil-moisture\new-code\results\run_20250210_143049


In [2]:
# --- Load configuration ---
cfg = load_config()

model_builders = {
    "Simple LSTM": build_original_simple_lstm_model,
    "Optimized LSTM": build_optimized_lstm_model,
    "CNN": build_cnn_model
}

In [3]:
# --- Load and preprocess data ---
station_data = load_station_data(cfg)
for station in station_data:
    print(f"Engineering features for {station} ...")
    station_data[station] = engineer_data(station_data[station])

# For forecasting, choose Station1 (for example)
df_forecast = station_data["Station1"]
scaled_df, scaler = scale_df(df_forecast)

Loaded data from 6 stations: ['Station1', 'Station2', 'Station3', 'Station4', 'Station5', 'Station6']
Engineering features for Station1 ...
Engineering features for Station2 ...
Engineering features for Station3 ...
Engineering features for Station4 ...
Engineering features for Station5 ...
Engineering features for Station6 ...


In [4]:
# --- Create Sliding Windows ---
TARGET_COLUMN = cfg["model"]["target_column"]  # "SWC_20"
WINDOW_SIZE = cfg["model"]["window_size"]         # 72
OFFSET = cfg["model"].get("offset", 24)             # 24
X, y = data_to_X_y(scaled_df[TARGET_COLUMN], WINDOW_SIZE, OFFSET)
print("Full sliding windows shapes: X =", X.shape, ", y =", y.shape)

Full sliding windows shapes: X = (52464, 72, 1) , y = (52464,)


In [5]:
# Split into training (70%), validation (20%), test (10%)
n = len(X)
X_train, y_train = X[:int(n*0.7)], y[:int(n*0.7)]
X_val, y_val = X[int(n*0.7):int(n*0.9)], y[int(n*0.7):int(n*0.9)]
X_test, y_test = X[int(n*0.9):], y[int(n*0.9):]
print("Split shapes -- X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)


Split shapes -- X_train: (36724, 72, 1) X_val: (10493, 72, 1) X_test: (5247, 72, 1)


In [6]:
# --- Build and Train Baseline Model ---
# Save checkpoint in the timestamped results folder
baseline_checkpoint = str(results_folder / "model1.keras")
model = build_original_simple_lstm_model(WINDOW_SIZE)
cp = tf.keras.callbacks.ModelCheckpoint(baseline_checkpoint, save_best_only=True, monitor='val_loss', mode='min')
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min')
EPOCHS = cfg["lstm"]["epochs"]
BATCH_SIZE = cfg["lstm"]["batch_size"]

history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE,
                    callbacks=[cp, early_stopping])

Epoch 1/10


c:\Users\chris\miniconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


1148/1148 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - loss: 0.0367 - root_mean_squared_error: 0.1763 - val_loss: 0.0035 - val_root_mean_squared_error: 0.0591
Epoch 2/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - loss: 0.0046 - root_mean_squared_error: 0.0681 - val_loss: 0.0034 - val_root_mean_squared_error: 0.0586
Epoch 3/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - loss: 0.0042 - root_mean_squared_error: 0.0644 - val_loss: 0.0033 - val_root_mean_squared_error: 0.0572
Epoch 4/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - loss: 0.0042 - root_mean_squared_error: 0.0651 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0569
Epoch 5/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - loss: 0.0039 - root_mean_squared_error: 0.0628 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0565
Epoch 6/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 23s 20ms/step - loss: 0.0038 - root_mean_squared_error: 0.0617 - val_loss: 0.0031 - val_root_mean_squared_error: 0.0558
Epoch 7/10
1148/1148 ━━━━━━━━━━━━━━━━━━

In [7]:
# --- Reload Best Model and Generate Predictions ---
model = tf.keras.models.load_model(baseline_checkpoint)
predictions_scaled = model.predict(X_test).flatten()


164/164 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [8]:
preds_inv_df = inverse_scale(predictions_scaled, scaled_df, TARGET_COLUMN)
predictions_rescaled = preds_inv_df[TARGET_COLUMN].values

actual_inv_df = inverse_scale(y_test.reshape(-1, 1), scaled_df, TARGET_COLUMN)
y_actual = actual_inv_df[TARGET_COLUMN].values
print("First 5 rescaled predictions:", predictions_rescaled[:5])

First 5 rescaled predictions: [0.80669826 0.81059146 0.81162781 0.81164491 0.80965918]


In [9]:
# --- Plot Loss and Predictions ---
loss_plot_path = results_folder / "loss_plot.png"
plot_loss(history, save_path=str(loss_plot_path))

predictions_plot_path = results_folder / "predictions_plot.png"
plot_predictions(y_actual, predictions_rescaled, TARGET_COLUMN, save_path=str(predictions_plot_path))


# --- Print Evaluation Metrics ---
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
print("R2 Score:", r2_score(y_actual, predictions_rescaled))
print("Mean Squared Error:", mean_squared_error(y_actual, predictions_rescaled))
print("Mean Absolute Error:", mean_absolute_error(y_actual, predictions_rescaled))
print("Mean Absolute Percentage Error:", mean_absolute_percentage_error(y_actual, predictions_rescaled))

Loss plot saved to c:\Users\chris\OneDrive\Documents\Academic\CS 370\tx-soil-moisture\new-code\results\run_20250210_143049\loss_plot.png
Predictions plot saved to c:\Users\chris\OneDrive\Documents\Academic\CS 370\tx-soil-moisture\new-code\results\run_20250210_143049\predictions_plot.png
R2 Score: 0.9077569832361463
Mean Squared Error: 0.0017836151974869518
Mean Absolute Error: 0.020011558559354714
Mean Absolute Percentage Error: 0.05748251140565345
